# Actividad 3 · GAC | Preparación de datos y regresión logística

**Propósito.** Limpiar seis bases, transformar variables dicotómicas y comparar modelos de clasificación.

**Cómo ejecutar:** colocar los seis CSV originales junto a este notebook y utilizar **Run All** en VS Code. Se generan carpetas `datos_privados`, `datos_procesados` y `resultados`. Los archivos privados y las bases crudas **no deben publicarse**.

**Alcance:** conserva los criterios del ejercicio original (umbrales 4%, 60%, 75%; semilla 42; separación 70/30). Los modelos son exploratorios. Las variables de etapas comerciales no necesariamente están disponibles al captar un lead.

## 1. Configuración y fuentes

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

carpeta = Path.cwd()
print('Carpeta de trabajo:', carpeta)

In [ ]:
import pandas as pd
from pathlib import Path

# Ubicación de los archivos
carpeta = Path.cwd()

# Cargar las bases originales
bitacora = pd.read_csv(
    carpeta / "Analisis GAC Full7agosto 2026(Bitacora de Piso)(4).csv"
)

leads = pd.read_csv(
    carpeta / "Analisis GAC Full7agosto 2026(Leads Reales)(5).csv"
)

print("BITÁCORA DE PISO")
print("Dimensiones:", bitacora.shape)
display(bitacora.head())

print("\nLEADS REALES")
print("Dimensiones:", leads.shape)
display(leads.head())

## 2. Bitácora de Piso · Limpieza y codificación
Se eliminan columnas totalmente vacías y registros sin nombre **y** fecha; se retienen variables binarias y valores faltantes. Los 7 conflictos `Estatus Lead` / `Venta` se conservan para auditoría.

In [ ]:
# Crear una copia para proteger los datos originales
bitacora_limpia = bitacora.copy()

# 1. Eliminar columnas completamente vacías
columnas_vacias = bitacora_limpia.columns[
    bitacora_limpia.isna().all()
]

bitacora_limpia = bitacora_limpia.drop(
    columns=columnas_vacias
)

# 2. Eliminar filas completamente vacías
bitacora_limpia = bitacora_limpia.dropna(
    how="all"
)

# 3. Convertir las variables booleanas a 0 y 1
columnas_binarias = [
    "PDM",
    "SDC",
    "Venta",
    "Inter Gerente"
]

def convertir_binaria(valor):
    if pd.isna(valor):
        return pd.NA

    if valor is True or str(valor).strip().lower() in ["true", "1", "si", "sí"]:
        return 1

    if valor is False or str(valor).strip().lower() in ["false", "0", "no"]:
        return 0

    return pd.NA

for columna in columnas_binarias:
    bitacora_limpia[columna] = (
        bitacora_limpia[columna]
        .apply(convertir_binaria)
        .astype("Int64")
    )

# 4. Revisar resultados
print("Dimensiones originales:", bitacora.shape)
print("Dimensiones después de limpieza:", bitacora_limpia.shape)

print("\nColumnas eliminadas:")
print(columnas_vacias.tolist())

print("\nDistribución de Venta:")
print(bitacora_limpia["Venta"].value_counts(dropna=False))

display(bitacora_limpia.head())

In [ ]:
# Corregir espacios en los nombres de las columnas
bitacora_limpia.columns = (
    bitacora_limpia.columns
    .str.strip()
)

# Identificar duplicados
duplicados = bitacora_limpia[
    bitacora_limpia.duplicated(keep=False)
]

print("Duplicados exactos:",
      bitacora_limpia.duplicated().sum())

print("\nPrimeros registros duplicados:")
display(duplicados.head(15))

# Comprobar filas sin identificación
print("\nRegistros sin nombre y sin fecha:")

print(
    bitacora_limpia[
        ["Nombre Cliente", "Fecha"]
    ].isna().all(axis=1).sum()
)

In [ ]:
# Crear copia antes de continuar
bitacora_limpia = bitacora_limpia.copy()

# Identificar registros sin nombre y sin fecha
sin_identificacion = (
    bitacora_limpia["Nombre Cliente"].isna() &
    bitacora_limpia["Fecha"].isna()
)

print("Registros sin identificación:",
      sin_identificacion.sum())

# Separar registros que vamos a excluir
registros_excluidos = bitacora_limpia[
    sin_identificacion
].copy()

# Conservar registros identificables
bitacora_limpia = bitacora_limpia[
    ~sin_identificacion
].copy()

# Reiniciar índices
bitacora_limpia.reset_index(drop=True, inplace=True)

print("\nRegistros restantes:", len(bitacora_limpia))

print("\nDuplicados restantes:",
      bitacora_limpia.duplicated().sum())

print("\nDistribución de Venta:")
print(bitacora_limpia["Venta"].value_counts(dropna=False))

In [ ]:
# Normalizar espacios en los nombres de columnas
bitacora_limpia.columns = (
    bitacora_limpia.columns
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Verificar nombres
print("Columnas disponibles:")
print(bitacora_limpia.columns.tolist())

# Revisar variables categóricas
columnas_revision = [
    "Asesor",
    "Estatus Lead",
    "Temperatura",
    "Estatus de SDC"
]

for columna in columnas_revision:
    if columna in bitacora_limpia.columns:
        print(f"\n===== {columna} =====")
        print(
            bitacora_limpia[columna].value_counts(
                dropna=False
            )
        )
    else:
        print(f"\nNo se encontró: {columna}")

In [ ]:
# Crear copia de seguridad
bitacora_preparada = bitacora_limpia.copy()

# Identificar columnas de texto
columnas_texto = bitacora_preparada.select_dtypes(
    include=["object", "string"]
).columns

# Limpiar espacios en los textos
for columna in columnas_texto:
    bitacora_preparada[columna] = (
        bitacora_preparada[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

print("ASESORES DESPUÉS DE LIMPIEZA:")
print(
    bitacora_preparada["Asesor"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nESTATUS LEAD:")
print(
    bitacora_preparada["Esatus Lead"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nTEMPERATURA:")
print(
    bitacora_preparada["Temperatura"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nESTATUS DE SDC:")
print(
    bitacora_preparada["Estatus de SDC"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
# Corregir errores en los nombres de columnas

bitacora_preparada = bitacora_preparada.rename(
    columns={
        "Esatus Lead": "Estatus Lead"
    }
)

# Verificar que quedó correctamente
print("Columnas disponibles:")
print(bitacora_preparada.columns.tolist())

print("\nCategorías de Estatus Lead:")
print(
    bitacora_preparada["Estatus Lead"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
# PASO 15: Normalizar Estatus Lead

bitacora_preparada["Estatus Lead"] = (
    bitacora_preparada["Estatus Lead"]
    .astype("string")
    .str.strip()
    .str.title()
)

print("ESTATUS LEAD CORREGIDO:")
print(
    bitacora_preparada["Estatus Lead"]
    .value_counts(dropna=False)
)

# Comparar Estatus Lead contra Venta
print("\nCOMPARACIÓN ESTATUS LEAD VS VENTA:")

tabla_ventas = pd.crosstab(
    bitacora_preparada["Estatus Lead"].fillna("Sin dato"),
    bitacora_preparada["Venta"],
    dropna=False
)

display(tabla_ventas)

# Identificar registros inconsistentes
inconsistencias = bitacora_preparada[
    (
        (bitacora_preparada["Estatus Lead"] == "Venta") &
        (bitacora_preparada["Venta"] == 0)
    ) |
    (
        (bitacora_preparada["Estatus Lead"] != "Venta") &
        (bitacora_preparada["Venta"] == 1)
    )
]

print("\nREGISTROS QUE NECESITAN REVISIÓN:")

display(
    inconsistencias[
        ["Fecha", "Nombre Cliente", "Asesor",
         "Estatus Lead", "Venta", "PDM", "SDC"]
    ]
)

In [ ]:
# PASO 16: Crear variables dicotómicas

# Copia de trabajo
bitacora_modelo = bitacora_preparada.copy()

# Convertir variables binarias a números
for columna in ["PDM", "SDC", "Inter Gerente", "Venta"]:
    bitacora_modelo[columna] = pd.to_numeric(
        bitacora_modelo[columna],
        errors="coerce"
    )

# Codificación one-hot de Estatus Lead
# Se conserva una categoría para cada estado,
# incluyendo los valores faltantes.

estatus_dummies = pd.get_dummies(
    bitacora_modelo["Estatus Lead"],
    prefix="Estatus",
    dummy_na=True,
    dtype=int
)

# Agregar nuevas columnas
bitacora_modelo = pd.concat(
    [bitacora_modelo, estatus_dummies],
    axis=1
)

# Mostrar solamente variables binarias
columnas_mostrar = [
    "PDM",
    "SDC",
    "Inter Gerente",
    "Venta"
] + estatus_dummies.columns.tolist()

display(bitacora_modelo[columnas_mostrar].head(10))

print("\nDimensiones:", bitacora_modelo.shape)
print("\nVariables dicotómicas creadas:")
print(estatus_dummies.columns.tolist())

In [ ]:
# PASO 17: Guardar las bases procesadas

from pathlib import Path

carpeta = Path.cwd()

# Base limpia con las categorías originales
bitacora_preparada.to_csv(
    carpeta / "Bitacora_Piso_Limpia.csv",
    index=False,
    encoding="utf-8-sig"
)

# Base con variables dicotómicas
bitacora_modelo.to_csv(
    carpeta / "Bitacora_Piso_Dicotomica.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos guardados correctamente:")
print("1. Bitacora_Piso_Limpia.csv")
print("2. Bitacora_Piso_Dicotomica.csv")

## 3. Leads Reales · Reconstrucción mensual y variables objetivo
Se toman encabezados de la segunda fila; se excluyen filas auxiliares y enero de 2024 sin indicadores. Umbrales exploratorios: conversión ≥4% y efectividad ≥60%.

In [ ]:
# PASO 19: Reconstruir los encabezados de Leads Reales

leads_limpia = leads.copy()

# La fila 1 contiene los encabezados reales
encabezados = leads_limpia.iloc[1].tolist()

# Asignar nombres correctos
leads_limpia.columns = encabezados

# Retirar las filas de resumen y encabezados
leads_limpia = leads_limpia.iloc[2:].copy()

# Normalizar nombres de columnas
leads_limpia.columns = (
    pd.Index(leads_limpia.columns)
    .map(lambda x: str(x).strip() if pd.notna(x) else "Sin_nombre")
)

print("COLUMNAS RECONSTRUIDAS:")
print(leads_limpia.columns.tolist())

print("\nPRIMEROS REGISTROS:")
display(leads_limpia.head(12))

print("\nDIMENSIONES:")
print(leads_limpia.shape)

In [ ]:
# PASO 20: Identificar los registros mensuales

# Crear copia de trabajo
leads_preparada = leads_limpia.copy()

# Eliminar columnas sin nombre
leads_preparada = leads_preparada.loc[
    :,
    leads_preparada.columns != "Sin_nombre"
].copy()

# Normalizar Año y Mes
for columna in ["Año", "Mes"]:
    leads_preparada[columna] = (
        leads_preparada[columna]
        .astype("string")
        .str.strip()
    )

# Convertir Año a formato numérico
leads_preparada["Año"] = pd.to_numeric(
    leads_preparada["Año"],
    errors="coerce"
)

# Identificar filas donde aparece un año
print("===== AÑOS IDENTIFICADOS =====")

display(
    leads_preparada.loc[
        leads_preparada["Año"].notna(),
        ["Año", "Mes", "Total", "Ventas"]
    ]
)

# Mostrar los valores de la columna Mes
print("\n===== VALORES EN LA COLUMNA MES =====")

print(
    leads_preparada["Mes"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
# PASO 21: Organizar los registros mensuales

# Copiar la base preparada
leads_mensuales = leads_preparada.copy()

# Completar los años faltantes
leads_mensuales["Año"] = leads_mensuales["Año"].ffill()

# Definir los meses válidos
meses_validos = [
    "Enero", "Febrero", "Marzo", "Abril",
    "Mayo", "Junio", "Julio", "Agosto",
    "Septiembre", "Octubre", "Noviembre",
    "Diciembre"
]

# Conservar únicamente filas de meses
leads_mensuales = leads_mensuales[
    leads_mensuales["Mes"].isin(meses_validos)
].copy()

# Definir indicadores principales
indicadores = [
    "Total", "Ilocalizable", "No Contesta",
    "D. Incorrectos", "Duplicados",
    "Efectivos", "Abiertos", "Ventas",
    "Conversion", "Efectividad de Leads"
]

# Eliminar meses sin ningún indicador
leads_mensuales = leads_mensuales.dropna(
    how="all",
    subset=indicadores
)

# Reiniciar índices
leads_mensuales.reset_index(
    drop=True,
    inplace=True
)

# Mostrar resultados
print("DIMENSIONES:")
print(leads_mensuales.shape)

print("\nREGISTROS POR AÑO:")
print(
    leads_mensuales["Año"]
    .value_counts()
    .sort_index()
)

print("\nPRIMEROS 15 REGISTROS:")
display(leads_mensuales.head(15))

print("\nÚLTIMOS 10 REGISTROS:")
display(leads_mensuales.tail(10))

In [ ]:
# PASO 22: Convertir variables numéricas

# Crear copia de seguridad
leads_numerica = leads_mensuales.copy()

# Columnas que deben ser numéricas
columnas_numericas = [
    "Total",
    "Ilocalizable",
    "No Contesta",
    "D. Incorrectos",
    "Duplicados",
    "Efectivos",
    "Abiertos",
    "Ventas"
]

# Eliminar separadores de miles y convertir a números
for columna in columnas_numericas:
    leads_numerica[columna] = pd.to_numeric(
        leads_numerica[columna]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

# Convertir porcentajes a valores decimales
columnas_porcentajes = [
    "Conversion",
    "Efectividad de Leads"
]

for columna in columnas_porcentajes:
    leads_numerica[columna] = (
        pd.to_numeric(
            leads_numerica[columna]
            .astype("string")
            .str.replace("%", "", regex=False)
            .str.strip(),
            errors="coerce"
        ) / 100
    )

# Convertir año a entero
leads_numerica["Año"] = (
    leads_numerica["Año"].astype(int)
)

print("TIPOS DE DATOS:")
print(leads_numerica.dtypes)

print("\nPRIMEROS REGISTROS:")
display(leads_numerica.head())

print("\nVALORES NULOS:")
print(leads_numerica.isna().sum())

In [ ]:
# PASO 23: Validar los datos numéricos

print("RESUMEN ESTADÍSTICO:")
display(leads_numerica.describe())

print("\nDUPLICADOS AÑO-MES:")
print(
    leads_numerica.duplicated(
        subset=["Año", "Mes"]
    ).sum()
)

print("\nVALORES NEGATIVOS:")
print(
    (leads_numerica[columnas_numericas] < 0)
    .sum()
)

print("\nVENTAS MAYORES QUE LEADS EFECTIVOS:")
print(
    (
        leads_numerica["Ventas"] >
        leads_numerica["Efectivos"]
    ).sum()
)

print("\nEFECTIVOS MAYORES QUE TOTAL:")
print(
    (
        leads_numerica["Efectivos"] >
        leads_numerica["Total"]
    ).sum()
)

In [ ]:
# PASO 24: Validar los porcentajes

leads_final = leads_numerica.copy()

# Calcular tasas a partir de las cantidades
leads_final["Conversion_Calculada"] = (
    leads_final["Ventas"] /
    leads_final["Efectivos"]
)

leads_final["Efectividad_Calculada"] = (
    leads_final["Efectivos"] /
    leads_final["Total"]
)

# Diferencias entre tasas registradas y calculadas
leads_final["Diferencia_Conversion"] = (
    leads_final["Conversion"] -
    leads_final["Conversion_Calculada"]
).abs()

leads_final["Diferencia_Efectividad"] = (
    leads_final["Efectividad de Leads"] -
    leads_final["Efectividad_Calculada"]
).abs()

# Mostrar comparación
columnas_revision = [
    "Año",
    "Mes",
    "Conversion",
    "Conversion_Calculada",
    "Diferencia_Conversion",
    "Efectividad de Leads",
    "Efectividad_Calculada",
    "Diferencia_Efectividad"
]

display(
    leads_final[columnas_revision].round(4)
)

# Detectar diferencias superiores a 1 punto porcentual
print("MESES CON DIFERENCIAS IMPORTANTES:")

diferencias = leads_final[
    (leads_final["Diferencia_Conversion"] > 0.01) |
    (leads_final["Diferencia_Efectividad"] > 0.01)
]

display(diferencias[columnas_revision].round(4))

print("Total de meses con diferencias:",
      len(diferencias))

In [ ]:
# PASO 25: Variables dicotómicas

leads_final["Alta_Conversion"] = (
    leads_final["Conversion_Calculada"] >= 0.04
).astype(int)

leads_final["Alta_Efectividad"] = (
    leads_final["Efectividad_Calculada"] >= 0.60
).astype(int)

# Convertir meses en variables dicotómicas
meses_dicotomicos = pd.get_dummies(
    leads_final["Mes"],
    prefix="Mes",
    dtype=int
)

# Agregar las variables nuevas
leads_final = pd.concat(
    [leads_final, meses_dicotomicos],
    axis=1
)

print("DISTRIBUCIÓN DE ALTA CONVERSIÓN:")
print(leads_final["Alta_Conversion"].value_counts())

print("\nDISTRIBUCIÓN DE ALTA EFECTIVIDAD:")
print(leads_final["Alta_Efectividad"].value_counts())

print("\nDIMENSIONES FINALES:")
print(leads_final.shape)

display(
    leads_final[
        ["Año", "Mes", "Alta_Conversion", "Alta_Efectividad"]
    ].head(15)
)

In [ ]:
# PASO 26: Guardar las bases de Leads Reales

from pathlib import Path

carpeta = Path.cwd()

# Base mensual limpia, sin columnas adicionales de modelado
leads_numerica.to_csv(
    carpeta / "Leads_Reales_Limpia.csv",
    index=False,
    encoding="utf-8-sig"
)

# Base con variables dicotómicas
leads_final.to_csv(
    carpeta / "Leads_Reales_Dicotomica.csv",
    index=False,
    encoding="utf-8-sig"
)

print("ARCHIVOS GUARDADOS CORRECTAMENTE")
print("1. Leads_Reales_Limpia.csv")
print("2. Leads_Reales_Dicotomica.csv")

## 4. Modelado de Leads Reales
Se reporta tanto la separación estratificada como una prueba temporal. **La prueba temporal de 2026 contiene solamente clase positiva**, por lo que no demuestra discriminación entre clases.

In [ ]:
# PASO 27: Preparar X e y

from sklearn.model_selection import train_test_split

# Variables predictoras
X = leads_final[
    ["Año", "Total"]
    + meses_dicotomicos.columns.tolist()
].copy()

# Variable objetivo
y = leads_final["Alta_Conversion"].copy()

# Convertir los datos a tipo numérico
X = X.astype(float)
y = y.astype(int)

# División estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("REGISTROS DE ENTRENAMIENTO:", len(X_train))
print("REGISTROS DE PRUEBA:", len(X_test))

print("\nCLASES EN ENTRENAMIENTO:")
print(y_train.value_counts())

print("\nCLASES EN PRUEBA:")
print(y_test.value_counts())

print("\nVARIABLES PREDICTORAS:")
print(X.columns.tolist())

In [ ]:
# PASO 28: Entrenar modelos de regresión logística

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# MODELO 1: Regresión logística normal
modelo_normal = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=42
    )
)

modelo_normal.fit(X_train, y_train)

y_pred_normal = modelo_normal.predict(X_test)

print("Modelo normal entrenado correctamente")

# MODELO 2: Regresión logística balanceada
modelo_balanceado = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

modelo_balanceado.fit(X_train, y_train)

y_pred_balanceado = modelo_balanceado.predict(X_test)

print("Modelo balanceado entrenado correctamente")

In [ ]:
# PASO 29: Evaluación de regresión logística

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import pandas as pd
import matplotlib.pyplot as plt

resultados = []

modelos = {
    "Regresión logística normal": y_pred_normal,
    "Regresión logística balanceada": y_pred_balanceado
}

for nombre, predicciones in modelos.items():

    resultados.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_test, predicciones),
        "Precisión": precision_score(
            y_test, predicciones, zero_division=0
        ),
        "Sensibilidad": recall_score(
            y_test, predicciones, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, predicciones, zero_division=0
        )
    })

    matriz = confusion_matrix(
        y_test,
        predicciones,
        labels=[0, 1]
    )

    visualizacion = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Baja", "Alta"]
    )

    visualizacion.plot(
        cmap="Blues",
        values_format="d"
    )

    plt.title(nombre)
    plt.show()

# Tabla comparativa
tabla_resultados = pd.DataFrame(resultados)

print("COMPARACIÓN DE MODELOS:")
display(tabla_resultados.round(3))

In [ ]:
# PASO 30: Validación cronológica

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import pandas as pd

# Ordenar los registros cronológicamente
meses_orden = {
    "Enero": 1, "Febrero": 2, "Marzo": 3,
    "Abril": 4, "Mayo": 5, "Junio": 6,
    "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Octubre": 10,
    "Noviembre": 11, "Diciembre": 12
}

datos_temporales = leads_final.copy()

datos_temporales["Numero_Mes"] = (
    datos_temporales["Mes"].map(meses_orden)
)

datos_temporales = datos_temporales.sort_values(
    ["Año", "Numero_Mes"]
)

# Usar las mismas variables predictoras
variables = X.columns.tolist()

# Entrenar con 2024 y 2025
train = datos_temporales[
    datos_temporales["Año"] < 2026
]

# Evaluar con 2026
test = datos_temporales[
    datos_temporales["Año"] == 2026
]

X_train_t = train[variables].astype(float)
y_train_t = train["Alta_Conversion"].astype(int)

X_test_t = test[variables].astype(float)
y_test_t = test["Alta_Conversion"].astype(int)

# Entrenar ambos modelos
resultados_temporales = []

for nombre, peso in [
    ("Normal", None),
    ("Balanceado", "balanced")
]:

    modelo = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight=peso,
            max_iter=1000,
            random_state=42
        )
    )

    modelo.fit(X_train_t, y_train_t)

    predicciones = modelo.predict(X_test_t)

    resultados_temporales.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(
            y_test_t, predicciones
        ),
        "Precisión": precision_score(
            y_test_t, predicciones,
            zero_division=0
        ),
        "Sensibilidad": recall_score(
            y_test_t, predicciones,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_test_t, predicciones,
            zero_division=0
        )
    })

    matriz = confusion_matrix(
        y_test_t,
        predicciones,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Baja", "Alta"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"Validación cronológica - {nombre}")
    plt.show()

print("RESULTADOS DE VALIDACIÓN CRONOLÓGICA:")

display(
    pd.DataFrame(resultados_temporales).round(3)
)

print("Meses de entrenamiento:", len(train))
print("Meses de prueba:", len(test))

In [ ]:
# PASO 31: Detección de outliers con IQR

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

variables_revision = [
    "Total",
    "Efectivos",
    "Ventas",
    "Ilocalizable",
    "No Contesta",
    "Abiertos"
]

resultados_outliers = []

for columna in variables_revision:

    Q1 = leads_numerica[columna].quantile(0.25)
    Q3 = leads_numerica[columna].quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    cantidad = (
        (leads_numerica[columna] < limite_inferior) |
        (leads_numerica[columna] > limite_superior)
    ).sum()

    resultados_outliers.append({
        "Variable": columna,
        "Límite inferior": limite_inferior,
        "Límite superior": limite_superior,
        "Outliers": cantidad
    })

tabla_outliers = pd.DataFrame(resultados_outliers)

print("RESULTADOS DE DETECCIÓN DE OUTLIERS:")
display(tabla_outliers.round(2))

# Visualizar valores atípicos
for columna in variables_revision:

    plt.figure(figsize=(8, 3))

    sns.boxplot(
        x=leads_numerica[columna]
    )

    plt.title(f"Boxplot - {columna}")
    plt.xlabel(columna)
    plt.tight_layout()
    plt.show()

In [ ]:
# PASO 32: Identificar meses con valores atípicos

variables_revision = [
    "Total",
    "Efectivos",
    "Ventas",
    "Ilocalizable",
    "No Contesta",
    "Abiertos"
]

registros_atipicos = []

for columna in variables_revision:

    Q1 = leads_numerica[columna].quantile(0.25)
    Q3 = leads_numerica[columna].quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    filtro = (
        (leads_numerica[columna] < limite_inferior) |
        (leads_numerica[columna] > limite_superior)
    )

    encontrados = leads_numerica.loc[
        filtro,
        ["Año", "Mes", columna]
    ].copy()

    encontrados["Variable"] = columna
    encontrados["Limite_Inferior"] = limite_inferior
    encontrados["Limite_Superior"] = limite_superior

    encontrados = encontrados.rename(
        columns={columna: "Valor"}
    )

    registros_atipicos.append(encontrados)

detalle_outliers = pd.concat(
    registros_atipicos,
    ignore_index=True
)

print("MESES CON VALORES ATÍPICOS:")

display(
    detalle_outliers[
        [
            "Año",
            "Mes",
            "Variable",
            "Valor",
            "Limite_Inferior",
            "Limite_Superior"
        ]
    ].round(2)
)

print("\nTotal de detecciones:",
      len(detalle_outliers))

In [ ]:
# PASO 33: Guardar diagnóstico de outliers

from pathlib import Path

carpeta = Path.cwd()

# Guardar detalle de valores atípicos
detalle_outliers.to_csv(
    carpeta / "Diagnostico_Outliers_GAC.csv",
    index=False,
    encoding="utf-8-sig"
)

# Documentar decisión
print("DIAGNÓSTICO DE VALORES ATÍPICOS")
print("--------------------------------")
print("Total de detecciones:", len(detalle_outliers))
print("Meses distintos afectados:",
      len(detalle_outliers[["Año", "Mes"]].drop_duplicates()))
print("Decisión: conservar los registros originales.")
print("Justificación: no se confirmaron errores de captura.")
print("Archivo guardado: Diagnostico_Outliers_GAC.csv")

In [ ]:
# PASO 34: Regresión logística para Alta Efectividad

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import pandas as pd
import matplotlib.pyplot as plt

# Variables predictoras
X_efectividad = leads_final[
    ["Año", "Total"]
    + meses_dicotomicos.columns.tolist()
].astype(float)

# Variable objetivo
y_efectividad = leads_final[
    "Alta_Efectividad"
].astype(int)

# Dividir datos
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_efectividad,
    y_efectividad,
    test_size=0.30,
    random_state=42,
    stratify=y_efectividad
)

# Entrenar modelos
resultados_efectividad = []

for nombre, peso in [
    ("Normal", None),
    ("Balanceado", "balanced")
]:

    modelo = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight=peso,
            max_iter=1000,
            random_state=42
        )
    )

    modelo.fit(X_train_e, y_train_e)

    predicciones = modelo.predict(X_test_e)

    resultados_efectividad.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_test_e, predicciones),
        "Precisión": precision_score(
            y_test_e, predicciones, zero_division=0
        ),
        "Sensibilidad": recall_score(
            y_test_e, predicciones, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test_e, predicciones, zero_division=0
        )
    })

    matriz = confusion_matrix(
        y_test_e,
        predicciones,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Baja", "Alta"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"Alta Efectividad - {nombre}")
    plt.show()

print("RESULTADOS DE ALTA EFECTIVIDAD:")

tabla_efectividad = pd.DataFrame(resultados_efectividad)

display(tabla_efectividad.round(3))

## 5. Cuatro bases adicionales · Lectura eficiente y limpieza
Se leen bloques grandes sin cargar filas vacías. Se mantienen estatus no confirmados como categorías originales y no se interpretan faltantes como ventas negativas.

In [ ]:
# PASO 34: Cargar y explorar las cuatro bases nuevas

import pandas as pd
from pathlib import Path

carpeta = Path.cwd()

archivos = {
    "Base de Leads": "Analisis GAC Full7agosto 2026(Base de Leads)(3).csv",
    "Base de Tlaxcala": "Analisis GAC Full7agosto 2026(Base de Tlaxcala)(3).csv",
    "Citas Digital": "Analisis GAC Full7agosto 2026(Citas Digital)(3).csv",
    "Plazas": "Analisis GAC Full7agosto 2026(Plazas)(3).csv"
}

bases_nuevas = {}

for nombre, archivo in archivos.items():

    ruta = carpeta / archivo

    # Lectura por bloques para evitar cargar
    # cientos de miles de filas vacías
    bloques = []

    for bloque in pd.read_csv(
        ruta,
        chunksize=20000,
        low_memory=False
    ):
        bloque.columns = (
            bloque.columns
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        # Conservar filas con un cliente identificable
        if nombre in ["Citas Digital"]:
            identificador = "Nombre Cliente"
        else:
            identificador = "Nombre"

        filtro = (
            bloque[identificador].notna() &
            bloque[identificador].astype(str).str.strip().ne("")
        )

        bloques.append(bloque.loc[filtro].copy())

    df = pd.concat(bloques, ignore_index=True)

    # Corregir la columna sin nombre de Base de Leads
    if nombre == "Base de Leads":
        if "Unnamed: 13" in df.columns:
            df = df.rename(
                columns={"Unnamed: 13": "Estatus"}
            )

        if "" in df.columns:
            df = df.rename(
                columns={"": "Estatus"}
            )

    # Eliminar columnas completamente vacías
    df = df.dropna(axis=1, how="all")

    bases_nuevas[nombre] = df

    print(f"\n========== {nombre} ==========")
    print("Dimensiones:", df.shape)
    print("Duplicados exactos:", df.duplicated().sum())

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nTipos de datos:")
    print(df.dtypes.to_string())

    print("\nPrimeros registros:")
    display(df.head(3))

In [ ]:
# PASO 35: Revisar variables para regresión logística

variables_revision = [
    "Estatus",
    "Estatus de Lead",
    "Temperatura",
    "Potencial de compra",
    "Seguimiento",
    "Cita efetiva",
    "PDM",
    "SDC",
    "Venta"
]

for nombre, df in bases_nuevas.items():

    print(f"\n========== {nombre} ==========")

    for columna in variables_revision:

        if columna in df.columns:

            print(f"\n--- {columna} ---")

            print(
                df[columna]
                .value_counts(dropna=False)
                .head(20)
                .to_string()
            )

In [ ]:
# PASO 36: Limpieza de Base de Leads y Tlaxcala

import pandas as pd

bases_leads_limpias = {}

for nombre in ["Base de Leads", "Base de Tlaxcala"]:

    df = bases_nuevas[nombre].copy()

    # Normalizar textos
    columnas_texto = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for columna in columnas_texto:
        df[columna] = (
            df[columna]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )

    # Normalizar estatus
    df["Estatus"] = df["Estatus"].str.title()

    # Convertir fecha
    df["Fecha de Alta"] = pd.to_datetime(
        df["Fecha de Alta"],
        errors="coerce"
    )

    # Crear variable dicotómica de venta
    df["Venta_Confirmada"] = (
        df["Estatus"] == "1 Ventas"
    ).astype(int)

    # Crear variables dicotómicas de temperatura
    temperatura_dummies = pd.get_dummies(
        df["Temperatura"],
        prefix="Temperatura",
        dummy_na=True,
        dtype=int
    )

    df = pd.concat(
        [df, temperatura_dummies],
        axis=1
    )

    # Guardar en diccionario
    bases_leads_limpias[nombre] = df

    print(f"\n========== {nombre} ==========")

    print("Dimensiones:", df.shape)

    print("\nVentas confirmadas:")
    print(df["Venta_Confirmada"].value_counts())

    print("\nValores nulos:")
    print(df.isna().sum().to_string())

    print("\nPrimeros registros:")
    display(df.head(3))

In [ ]:
# PASO 37: Diagnóstico de Citas Digital y Plazas

for nombre in ["Citas Digital", "Plazas"]:

    df = bases_nuevas[nombre].copy()

    print(f"\n========== {nombre} ==========")

    diagnostico = pd.DataFrame({
        "Columna": df.columns,
        "Nulos": df.isna().sum().values,
        "Porcentaje_Nulos": (
            df.isna().mean().values * 100
        ).round(2),
        "Valores_Unicos": df.nunique(dropna=True).values
    })

    display(diagnostico)

    variables = [
        "Estatus de Lead",
        "Estatus",
        "Seguimiento",
        "Cita efetiva",
        "PDM",
        "SDC",
        "Venta",
        "Potencial de compra"
    ]

    for columna in variables:

        if columna in df.columns:

            print(f"\n--- {columna} ---")

            print(
                df[columna]
                .value_counts(dropna=False)
                .head(20)
                .to_string()
            )

In [ ]:
# PASO 38: Limpieza y transformación dicotómica

import pandas as pd
import numpy as np

bases_comerciales_limpias = {}

def convertir_binaria(valor):
    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip().lower()

    if texto in ["true", "1", "sí", "si", "verdadero"]:
        return 1

    if texto in ["false", "0", "no", "falso"]:
        return 0

    return pd.NA


for nombre in ["Citas Digital", "Plazas"]:

    df = bases_nuevas[nombre].copy()

    # Limpiar nombres de columnas
    df.columns = (
        df.columns
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Limpiar espacios en las columnas de texto
    columnas_texto = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for columna in columnas_texto:
        df[columna] = (
            df[columna]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )

    # Identificar columnas binarias
    columnas_binarias = [
        "Seguimiento",
        "Cita efetiva",
        "PDM",
        "SDC",
        "Venta"
    ]

    for columna in columnas_binarias:

        if columna in df.columns:

            # Conservar el contenido original para auditoría
            df[f"{columna}_Original"] = df[columna]

            df[columna] = (
                df[columna]
                .apply(convertir_binaria)
                .astype("Int64")
            )

    # Normalizar algunos errores evidentes de estatus
    columna_estatus = (
        "Estatus de Lead"
        if nombre == "Citas Digital"
        else "Estatus"
    )

    df[columna_estatus] = (
        df[columna_estatus]
        .astype("string")
        .str.strip()
        .str.title()
    )

    correcciones = {
        "Acitvo": "Activo",
        "Acrivo": "Activo",
        "Ativo": "Activo",
        "Activo–": "Activo",
        "Contacto": "Contactado"
    }

    df[columna_estatus] = (
        df[columna_estatus].replace(correcciones)
    )

    # Convertir potencial de compra a decimal
    if "Potencial de compra" in df.columns:

        df["Potencial_Compra_Num"] = pd.to_numeric(
            df["Potencial de compra"]
            .astype("string")
            .str.replace("%", "", regex=False)
            .str.replace(",", ".", regex=False),
            errors="coerce"
        ) / 100

    # Guardar resultado
    bases_comerciales_limpias[nombre] = df

    print(f"\n========== {nombre} ==========")

    print("Dimensiones:", df.shape)

    print("\nEstatus corregidos:")
    print(
        df[columna_estatus]
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nVariables binarias:")
    for columna in columnas_binarias:
        if columna in df.columns:
            print(f"\n{columna}:")
            print(df[columna].value_counts(dropna=False))

    print("\nPrimeros registros:")
    display(df.head(3))

In [ ]:
# PASO 39: Verificación final de las dos bases

for nombre, df in bases_comerciales_limpias.items():

    print(f"\n========== {nombre} ==========")

    if nombre == "Citas Digital":
        print("\nDistribución de ventas:")
        print(df["Venta"].value_counts(dropna=False))

        print("\nEstatus de registros con Venta = 1:")
        print(
            df.loc[df["Venta"] == 1, "Estatus de Lead"]
            .value_counts(dropna=False)
        )

        print("\nPotencial de compra:")
        print(df["Potencial_Compra_Num"].describe())

    else:
        print("\nRegistros facturados:")
        print(
            df["Estatus"]
            .eq("Facturado")
            .sum()
        )

        print("\nDistribución de cita efectiva:")
        print(
            df["Cita efetiva"]
            .value_counts(dropna=False)
        )

    print("\nPorcentaje de nulos por columna:")
    print(
        (df.isna().mean() * 100)
        .round(2)
        .to_string()
    )

In [ ]:
# PASO 40: Corrección final de estatus

# Citas Digital
citas_limpia = bases_comerciales_limpias["Citas Digital"].copy()

citas_limpia["Estatus de Lead"] = (
    citas_limpia["Estatus de Lead"]
    .replace({
        "Activo–": "Activo",
        "Activio": "Activo"
    })
)

# Plazas
plazas_limpia = bases_comerciales_limpias["Plazas"].copy()

plazas_limpia["Estatus"] = (
    plazas_limpia["Estatus"]
    .replace({
        "Contactato": "Contactado"
    })
)

print("ESTATUS CITAS DIGITAL:")
print(
    citas_limpia["Estatus de Lead"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nESTATUS PLAZAS:")
print(
    plazas_limpia["Estatus"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
# PASO 41: Crear variables dicotómicas adicionales

# CITAS DIGITAL

citas_limpia["Potencial_Alto"] = (
    citas_limpia["Potencial_Compra_Num"] >= 0.75
).astype("Int64")

# Los valores faltantes deben seguir siendo faltantes
citas_limpia.loc[
    citas_limpia["Potencial_Compra_Num"].isna(),
    "Potencial_Alto"
] = pd.NA

estatus_citas = pd.get_dummies(
    citas_limpia["Estatus de Lead"],
    prefix="Estatus",
    dummy_na=True,
    dtype=int
)

citas_limpia = pd.concat(
    [citas_limpia, estatus_citas],
    axis=1
)

# PLAZAS

plazas_limpia["Facturado_Confirmado"] = (
    plazas_limpia["Estatus"] == "Facturado"
).astype("Int64")

plazas_limpia.loc[
    plazas_limpia["Estatus"].isna() |
    plazas_limpia["Estatus"].eq("False"),
    "Facturado_Confirmado"
] = pd.NA

estatus_plazas = pd.get_dummies(
    plazas_limpia["Estatus"],
    prefix="Estatus",
    dummy_na=True,
    dtype=int
)

plazas_limpia = pd.concat(
    [plazas_limpia, estatus_plazas],
    axis=1
)

print("CITAS DIGITAL:")
print("Dimensiones:", citas_limpia.shape)
print("\nPotencial alto:")
print(citas_limpia["Potencial_Alto"].value_counts(dropna=False))

print("\nPLAZAS:")
print("Dimensiones:", plazas_limpia.shape)
print("\nFacturado confirmado:")
print(
    plazas_limpia["Facturado_Confirmado"]
    .value_counts(dropna=False)
)

In [ ]:
# PASO 42: Revisión final

# Normalizar variantes de Activo
citas_limpia["Estatus de Lead"] = (
    citas_limpia["Estatus de Lead"]
    .astype("string")
    .str.replace(r"^Activo\s*[-–—]\s*$", "Activo", regex=True)
)

# Verificar corrección
print("ESTATUS CITAS DIGITAL:")
print(
    citas_limpia["Estatus de Lead"]
    .value_counts(dropna=False)
    .to_string()
)

# Comprobar ventas
print("\nVENTAS CITAS DIGITAL:")
print(citas_limpia["Venta"].value_counts(dropna=False))

# Comprobar facturación
print("\nFACTURACIÓN PLAZAS:")
print(
    plazas_limpia["Facturado_Confirmado"]
    .value_counts(dropna=False)
)

In [ ]:
# PASO 43: Guardar las bases procesadas

from pathlib import Path

carpeta = Path.cwd()

bases_para_guardar = {
    "Base_Leads": bases_leads_limpias["Base de Leads"],
    "Base_Tlaxcala": bases_leads_limpias["Base de Tlaxcala"],
    "Citas_Digital": citas_limpia,
    "Plazas": plazas_limpia
}

# Directorios de salida
carpeta_privada = carpeta / "datos_privados"
carpeta_publica = carpeta / "datos_procesados"

carpeta_privada.mkdir(exist_ok=True)
carpeta_publica.mkdir(exist_ok=True)

for nombre, df in bases_para_guardar.items():

    # Guardar base completa localmente
    df.to_csv(
        carpeta_privada / f"{nombre}_Limpia.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Excluir información personal y texto libre
    columnas_excluir = [
        c for c in df.columns
        if (
            "nombre" in c.lower()
            or "telefono" in c.lower()
            or "comentario" in c.lower()
            or "aglead" in c.lower()
            or "asesor" in c.lower()
            or "¿por que" in c.lower()
            or "¿por qué" in c.lower()
            or "unnamed" in c.lower()
        )
    ]

    df_publico = df.drop(
        columns=columnas_excluir,
        errors="ignore"
    )

    df_publico.to_csv(
        carpeta_publica / f"{nombre}_Procesada.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{nombre}: archivos guardados")

print("\nProceso terminado.")

## 6. Citas Digital · Regresión logística, reponderación y SMOTE
Variable objetivo `Venta`. Predictores `Potencial_Compra_Num`, `PDM`, `SDC`. Imputación y estandarización dentro de cada pipeline. SMOTE solo actúa sobre entrenamiento; con variables binarias se trata de una comparación académica.

In [ ]:
# PASO 44: Preparar Citas Digital para regresión logística

from sklearn.model_selection import train_test_split
import pandas as pd

# Copia de trabajo
citas_modelo = citas_limpia.copy()

# Conservar registros con resultado conocido
citas_modelo = citas_modelo[
    citas_modelo["Venta"].notna()
].copy()

# Variable objetivo
y_citas = citas_modelo["Venta"].astype(int)

# Variables predictoras
variables_citas = [
    "Potencial_Compra_Num",
    "PDM",
    "SDC"
]

X_citas = citas_modelo[variables_citas].astype(float)

# División estratificada
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_citas,
    y_citas,
    test_size=0.30,
    random_state=42,
    stratify=y_citas
)

print("REGISTROS TOTALES:", len(citas_modelo))
print("ENTRENAMIENTO:", len(X_train_c))
print("PRUEBA:", len(X_test_c))

print("\nDISTRIBUCIÓN ORIGINAL:")
print(y_citas.value_counts())

print("\nCLASES EN ENTRENAMIENTO:")
print(y_train_c.value_counts())

print("\nCLASES EN PRUEBA:")
print(y_test_c.value_counts())

print("\nVALORES NULOS EN PREDICTORES:")
print(X_citas.isna().sum())

In [ ]:
# PASO 45: Entrenar tres modelos

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# MODELO 1: NORMAL
modelo_citas_normal = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_citas_normal.fit(X_train_c, y_train_c)

# MODELO 2: BALANCEADO
modelo_citas_balanceado = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler()),
    ("modelo", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

modelo_citas_balanceado.fit(X_train_c, y_train_c)

# MODELO 3: SMOTE
modelo_citas_smote = ImbPipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler()),
    ("smote", SMOTE(
        random_state=42,
        k_neighbors=5
    )),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_citas_smote.fit(X_train_c, y_train_c)

print("MODELOS ENTRENADOS CORRECTAMENTE")
print("1. Regresión logística normal")
print("2. Regresión logística balanceada")
print("3. Regresión logística con SMOTE")

In [ ]:
# PASO 46: Evaluación de los tres modelos

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import pandas as pd
import matplotlib.pyplot as plt

modelos_citas = {
    "Normal": modelo_citas_normal,
    "Balanceado": modelo_citas_balanceado,
    "SMOTE": modelo_citas_smote
}

resultados_citas = []

for nombre, modelo in modelos_citas.items():

    predicciones = modelo.predict(X_test_c)

    resultados_citas.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(
            y_test_c, predicciones
        ),
        "Exactitud balanceada": balanced_accuracy_score(
            y_test_c, predicciones
        ),
        "Precisión": precision_score(
            y_test_c, predicciones,
            zero_division=0
        ),
        "Sensibilidad": recall_score(
            y_test_c, predicciones,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_test_c, predicciones,
            zero_division=0
        )
    })

    matriz = confusion_matrix(
        y_test_c,
        predicciones,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Sin venta", "Venta"]
    ).plot(
        cmap="Blues",
        values_format="d"
    )

    plt.title(f"Regresión logística - {nombre}")
    plt.tight_layout()
    plt.show()

tabla_citas = pd.DataFrame(resultados_citas)

print("COMPARACIÓN DE LOS TRES MODELOS:")
display(tabla_citas.round(3))

In [ ]:
# PASO 47: Comparación con modelo de referencia

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Modelo de referencia: siempre predice no venta
pred_base = np.zeros(len(y_test_c), dtype=int)

modelos_predicciones = {
    "Referencia - Sin venta": pred_base,
    "Normal": modelo_citas_normal.predict(X_test_c),
    "Balanceado": modelo_citas_balanceado.predict(X_test_c),
    "SMOTE": modelo_citas_smote.predict(X_test_c)
}

comparacion_final = []

for nombre, pred in modelos_predicciones.items():

    comparacion_final.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_test_c, pred),
        "Exactitud balanceada": balanced_accuracy_score(
            y_test_c, pred
        ),
        "Precisión": precision_score(
            y_test_c, pred, zero_division=0
        ),
        "Sensibilidad": recall_score(
            y_test_c, pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test_c, pred, zero_division=0
        )
    })

tabla_comparacion_final = pd.DataFrame(comparacion_final)

display(tabla_comparacion_final.round(3))

In [ ]:
# PASO 48: Guardar resultados de clasificación

from pathlib import Path

carpeta = Path.cwd()
carpeta_resultados = carpeta / "resultados"

carpeta_resultados.mkdir(exist_ok=True)

# Tabla comparativa
tabla_comparacion_final.to_csv(
    carpeta_resultados / "Comparacion_Modelos_Citas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Resultados anteriores de Leads Reales
tabla_resultados.to_csv(
    carpeta_resultados / "Modelos_Alta_Conversion.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_efectividad.to_csv(
    carpeta_resultados / "Modelos_Alta_Efectividad.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados guardados correctamente.")

## 7. Resumen de verificación y recomendaciones

In [ ]:
resumen = pd.DataFrame([
    ('Bitácora de Piso', len(bitacora_modelo), bitacora_modelo.shape[1]),
    ('Leads Reales', len(leads_final), leads_final.shape[1]),
    ('Base de Leads', len(bases_leads_limpias['Base de Leads']), bases_leads_limpias['Base de Leads'].shape[1]),
    ('Base de Tlaxcala', len(bases_leads_limpias['Base de Tlaxcala']), bases_leads_limpias['Base de Tlaxcala'].shape[1]),
    ('Citas Digital', len(citas_limpia), citas_limpia.shape[1]),
    ('Plazas', len(plazas_limpia), plazas_limpia.shape[1])
], columns=['Base', 'Filas', 'Columnas'])
display(resumen)
print('Resultados Citas Digital:')
display(tabla_comparacion_final.round(3))
print('Archivos generados en:', carpeta)


### Criterios y límites metodológicos
- **Conservar nulos desconocidos** en variables binarias; imputarlos dentro del entrenamiento cuando el modelo lo requiere.
- **No interpretar el estatus `Activo`/`Contactado` como ventas perdidas.** En Base de Leads y Tlaxcala el `0` significa que la etiqueta no registra `1 Ventas`.
- **Valores atípicos:** se conservan los meses del análisis IQR de Leads Reales, sin evidencia de error de captura.
- **Desbalance:** comparar exactitud con sensibilidad, precisión, F1 y referencia que predice siempre no venta.
- **Privacidad:** los archivos del directorio `datos_procesados` son copias con algunas columnas identificadoras retiradas, **no una anonimización certificada**. Revisarlos antes de compartir/publicar; conservar los originales y `datos_privados` fuera de GitHub.
- **No combinar directamente las seis bases como si fueran la misma unidad de observación.**